# Mutation → Mechanism → Therapy — Main Run Notebook

**Repo:** [github.com/lightflow16/mutation-mechanism-therapy-amd](https://github.com/lightflow16/mutation-mechanism-therapy-amd)

**Setup on AMD notebook (run once in a terminal):**
```bash
cd /workspace
git clone --depth 1 https://github.com/lightflow16/mutation-mechanism-therapy-amd.git
cd mutation-mechanism-therapy-amd
```

Then run this notebook top-to-bottom. Python modules live in `src/`; outputs persist in `/workspace/shared/`.

## 0 — Confirm repo root (skip if you already `cd` into the clone)

In [ ]:
# Clone once in terminal (not here), then open this notebook from the repo root:
#   cd /workspace && git clone --depth 1 https://github.com/lightflow16/mutation-mechanism-therapy-amd.git
#   cd mutation-mechanism-therapy-amd

from pathlib import Path
ROOT = Path('.').resolve()
assert (ROOT / 'src' / 'pipeline.py').exists(), (
    'Run from repo root. Clone: git clone https://github.com/lightflow16/mutation-mechanism-therapy-amd.git'
)
print('ROOT =', ROOT)

## 1 — Environment + paths (CPU ok, no GPU yet)

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('.').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ['HF_HOME'] = '/workspace/shared/hf_cache'
os.environ['METRICS_DIR'] = '/workspace/shared/metrics'
Path('/workspace/shared/metrics').mkdir(parents=True, exist_ok=True)
Path('/workspace/shared/hf_cache').mkdir(parents=True, exist_ok=True)

import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

In [ ]:
!pip install -q -r requirements.txt openai

In [ ]:
!bash scripts/setup_external.sh

## 2 — Instant demo (cached traces, NO GPU, NO vLLM)

In [ ]:
from src.pipeline import run_case
import json

for gene, mut in [('EGFR','L858R'), ('PIK3CA','E545K'), ('TP53','R175H')]:
    r = run_case(gene, mut, architecture='blackboard', use_cached_trace=True, live_evidence=False)
    tr = r['reasoning'].get('target_reasoning', r['reasoning'])
    print(gene, mut, '->', r['route'], '| therapies:', tr.get('therapy',{}).get('sensitivity'))

## 3 — Attach GPU ONLY from here onward

Attach GPU in the AMD dashboard, then run the cells below back-to-back. Detach when idle.

In [ ]:
import time
GPU_ATTACH_T0 = time.time()  # budget proxy (~240 min)
print('GPU attached at', GPU_ATTACH_T0)

In [ ]:
# Start vLLM servers (run in a separate terminal OR background)
# !bash scripts/start_vllm.sh
from src.serving import check_all_endpoints
from src.config import load_config
print(check_all_endpoints(load_config()['serving']['endpoints']))

In [ ]:
# Live blackboard (needs vLLM). Skip if servers not up — cached demo above is submission-safe.
# r = run_case('EGFR','L858R', architecture='blackboard', use_cached_trace=False)
# print(json.dumps(r['reasoning'], indent=2)[:2000])

In [ ]:
# LoRA training (~10-30 min GPU)
# !python train/build_dataset.py
# !python train/lora_sft.py

In [ ]:
# TP53 rescue branch (ThermoMPNN + ESMFold — GPU)
# from src.pipeline import run_case
# r = run_case('TP53','R175H', architecture='blackboard', use_cached_trace=False, run_rescue_branch=True)
# print(r.get('rescue'))

In [ ]:
# Gradio demo (can run on CPU with cached traces)
# !python app.py

## 4 — Download submission artifacts

In [ ]:
!tar -czf /workspace/shared/submission_bundle.tgz \
  metrics/ data/traces/ deck/ \
  2>/dev/null || tar -czf /workspace/shared/submission_bundle.tgz -C /workspace/shared metrics data/traces deck 2>/dev/null || echo 'Adjust paths if needed'
!ls -lh /workspace/shared/*.tgz /workspace/shared/metrics/ 2>/dev/null | head